# Week 5 — Subqueries and Window Functions: Ranking, Running Totals, Row Numbers
## Phase 2b SQL | PORA Academy Cohort 7 — **Exercises**

Wednesday you used subqueries to compare a row against an aggregate — like finding orders priced above the average. Today's exercises push further: you'll combine a subquery with a window function, collapse a fan-out table before joining it, build a running total across months, and use `ROW_NUMBER()` to number rows inside a group.

Each question below comes as **three cells**:

1. A **question** with the task and an **Expected** result.
2. A blank `%%sql` answer cell — write your query where it says `-- Your query here`, capturing the result into the named variable (e.g. `q1`).
3. A **check cell** (plain Python) — run it after your query. A ✅ means you got it right, and the query result is displayed underneath.

**Do not edit the check cells** — they are locked and seeded from the real, verified Olist data.

⚠️ **Fan-out reminder:** `order_items`, `order_payments`, and `order_reviews` all hold **multiple rows per `order_id`**. If a question needs you to join more than one of them (or join one of them to a table you then aggregate), collapse the extra table to **one row per `order_id` in a `WITH` CTE first** — otherwise your totals/averages will be inflated by duplicate rows.

### Setup — run this cell first
Loads the Olist tables into a file-based SQLite database and connects `%%sql`. Because `autopandas` is on, every `%%sql` result is a pandas DataFrame your check cells can inspect.

In [ ]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


## Question 1 — Sellers above average seller revenue

Using a **subquery**, find every seller whose total revenue (`SUM(price)` from `order_items`) is above the *average* seller revenue across all sellers. Return one row per qualifying seller with `seller_id` and their `total_revenue` (rounded to 2 decimals).

`order_items` is a fan-out table (many rows per `order_id`, but that's fine here — you're grouping by `seller_id`, not `order_id`, so no collapsing CTE is needed for this one).

**Expected:** 628 sellers (out of 3,095 total sellers) have total revenue above the seller-wide average of R$4,391.48.

In [ ]:
%%sql q1 <<
-- Your query here

In [ ]:
# --- CHECK Q1 — do not edit ---
assert q1.shape[0] == 628, "Q1: expected 628 sellers above average seller revenue"
print("✅ Q1 correct")
q1  # show the result of your query

## Question 2 — Rank customer states by average review score

Rank every `customer_state` by its average review score, highest first. Join `orders`, `customers`, and `order_reviews`.

⚠️ `order_reviews` has 99,224 rows but only 98,673 distinct `order_id` (547 orders carry more than one review row). If you join it to `orders`/`customers` directly and then `AVG`, you will double-count those orders. **Collapse `order_reviews` to one row per `order_id` in a `WITH` CTE first** (`AVG(review_score) GROUP BY order_id`), *then* join that CTE to `orders`/`customers`, *then* `GROUP BY customer_state` and rank with `RANK() OVER (ORDER BY AVG(...) DESC)`.

Return `customer_state`, `avg_review_score` (rounded to 2 decimals), and `state_rank`, ordered by rank.

**Expected:** 27 states. Top-ranked state is **AM**, with an average review score of **4.21** (state_rank 1).

In [ ]:
%%sql q2 <<
-- Your query here

In [ ]:
# --- CHECK Q2 — do not edit ---
assert q2.shape[0] == 27, "Q2: expected 27 customer states"
assert q2.iloc[0]["customer_state"] == "AM", "Q2: top state by avg review score should be AM"
assert round(float(q2.iloc[0]["avg_review_score"]), 2) == 4.21, "Q2: expected AM avg review score of 4.21"
assert int(q2.iloc[0]["state_rank"]) == 1, "Q2: AM should be state_rank 1"
print("✅ Q2 correct")
q2  # show the result of your query

## Question 3 — Running total of payment revenue through 2018

Using `orders` joined to `order_payments`, compute total `payment_value` per month for every month in 2018 (`month` as `'YYYY-MM'` via `strftime`), plus a running total across months using `SUM(...) OVER (ORDER BY month)`.

This joins `orders` (the safe base table) to exactly **one** fan-out table (`order_payments`), so summing `payment_value` per month is safe — no collapsing CTE needed here.

Return `month`, `monthly_revenue` (rounded to 2 decimals), and `running_total` (rounded to 2 decimals), ordered by month.

**Expected:** 10 months of 2018 data. January 2018 revenue is R$1,115,004.18. The final month (October 2018) has a running total of R$8,699,763.05.

In [ ]:
%%sql q3 <<
-- Your query here

In [ ]:
# --- CHECK Q3 — do not edit ---
assert q3.shape[0] == 10, "Q3: expected 10 months of 2018 payment data"
assert q3.iloc[0]["month"] == "2018-01", "Q3: first month should be 2018-01"
assert round(float(q3.iloc[0]["monthly_revenue"]), 2) == 1115004.18, "Q3: expected Jan 2018 revenue R$1,115,004.18"
assert round(float(q3.iloc[-1]["running_total"]), 2) == 8699763.05, "Q3: expected final running total R$8,699,763.05"
print("✅ Q3 correct")
q3  # show the result of your query

## Question 4 — Number each review within its score group

Use `ROW_NUMBER()` to number every row of `order_reviews`, restarting the count at 1 within each `review_score` group (`PARTITION BY review_score`, `ORDER BY review_id`). Return `review_id`, `review_score`, and `row_num` for all rows, ordered by `review_score, row_num`.

**Expected:** 99,224 numbered rows total (one per review). Within the 5-star group, the row numbers run from 1 up to **57,328** (the total count of 5-star reviews) — the highest `row_num` in any partition always equals that partition's row count.

In [ ]:
%%sql q4 <<
-- Your query here

In [ ]:
# --- CHECK Q4 — do not edit ---
assert q4.shape[0] == 99224, "Q4: expected 99,224 numbered review rows"
five_star_max = int(q4.loc[q4["review_score"] == 5, "row_num"].max())
assert five_star_max == 57328, "Q4: expected row_num to reach 57,328 within the 5-star group"
print("✅ Q4 correct")
q4  # show the result of your query

## Wrap-up

You have now used subqueries (Wednesday) and window functions (Thursday) together: a scalar subquery to define "above average," a fan-out-safe CTE before ranking states, a running total with `SUM() OVER`, and `ROW_NUMBER()` to number rows within a partition without collapsing them. This is the same toolkit you will reach for in next week's multi-table joins and complex aggregations.
